# SPARC-P Agent Training Notebook

## 1.0 Introduction

This notebook trains SPARC-P agents on HiPerGator using **conda environments** (per UF RC requirements).

### 1.1 Environment Setup

**Before running this notebook, create the conda environment:**

```bash
cd /blue/jasondeanarnold/SPARCP
module load conda

# Create environment (first time only)
conda env create -f environment_training.yml -p /blue/jasondeanarnold/SPARCP/conda_envs/sparc_training

# Activate environment
# conda activate /blue/jasondeanarnold/SPARCP/conda_envs/sparc_training
```

**Note:** Python 3.11 is required for compatibility with CUDA 12.8 and PyTorch 2.1+.

### 1.2 Architectural Philosophy
This system uses a hybrid approach:
- **RAG (Retrieval-Augmented Generation)**: Provides real-time, factually accurate knowledge from the `/blue` storage tier.
- **PEFT/QLoRA**: Adapts a selected base model using 4-bit quantization. This notebook supports both `meta-llama/Llama-2-7b-hf` and `openai/gpt-oss-20b`.

### 1.3 Target Environment
- **System**: HiPerGator AI SuperPOD (NVIDIA A100/B200)
- **Package Manager**: Conda (mandatory per UF RC)
- **Storage**: `/blue` tier (home directory is strictly limited)

### 1.3 Architecture Diagram
![1.3 Architecture Diagram](../images/notebook1-1-3.png)

1.3 Architecture Diagram: This diagram details the full hybrid hardware and software architecture on HiPerGator, highlighting the dual-track system for real-time factual grounding (RAG) and persona adaptation (PEFT/QLoRA).

This is the environment setup cell â€” it loads every Python library the training pipeline needs and then confirms the environment is healthy before you proceed.

Specifically:
- Imports core Python utilities (`os`, `json`, `Path`) and then imports the heavy ML libraries: `datasets` (HuggingFace data loading), `transformers` (model loading and training), `peft` (LoRA adapter training), `trl` (the Supervised Fine-Tuning trainer), `langchain` and `langchain_chroma` (RAG retrieval), and `presidio` (PII anonymization).
- Prints the exact Python interpreter path and version so you can confirm you're in the correct `sparc_training` conda environment (not a system Python).
- Runs a `try/except` block that imports `torch`, checks GPU availability (`torch.cuda.is_available()`), and reports the PyTorch version. If any package is missing, instead of crashing silently it prints **exactly which conda commands to run** to create and activate the correct environment.

> **If you see "ERROR: Missing package":** Follow the printed instructions to create the conda environment from `environment_training.yml`. This only needs to be done once per HiPerGator account.

In [ ]:
# 2.3 Consolidated Imports and Environment Check

# IMPORTANT: On HiPerGator, use conda instead of pip (UF RC requirement)
# This notebook assumes the conda environment is already activated

import sys
import os
import json
from pathlib import Path
from typing import List, Dict, Optional

from datasets import load_dataset, Dataset
from pydantic import BaseModel, Field, ValidationError
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

print(f"Python: {sys.executable}")
print(f"Python version: {sys.version}")

# Verify key packages are available
try:
    import torch
    import transformers
    import peft
    import trl
    import bitsandbytes
    print(f"\n? All training packages available")
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError as e:
    base_path = os.environ.get("SPARC_BASE_PATH", "/blue/jasondeanarnold/SPARCP")
    print(f"\nERROR: Missing package - {e}")
    print("\nTo create the environment, run this ONCE on HiPerGator:")
    print("  module load conda")
    print(f"  conda env create -f environment_training.yml -p {base_path}/conda_envs/sparc_training")
    print("\nThen activate before running this notebook:")
    print("  module load conda")
    print(f"  conda activate {base_path}/conda_envs/sparc_training")

![2.1 System Configuration Diagram](../images/notebook1-2-1.png)

2.1 System Configuration Diagram: This flowchart outlines the strict environment setup and initialization sequence required on the UF HiPerGator system, prioritizing Conda and /blue storage over standard methods.

All central configuration values for training are defined here: storage paths, model selection, LoRA hyperparameters, and training arguments. This cell acts as the control panel for the notebook.

Key settings defined here:
- **`BASE_PATH`**: Root project directory on HiPerGator `/blue` storage (overridable via `SPARC_BASE_PATH`).
- **`OUTPUT_DIR`** and **`DATA_DIR`**: Output location for adapters and input location for training data.
- **`MODEL_NAME`**: Active base model for a single-model run.
- **`AVAILABLE_BASE_MODELS`**: Canonical list of supported trainable base models for this notebook:
  - `meta-llama/Llama-2-7b-hf`
  - `openai/gpt-oss-20b`
- **`COMPARE_MODEL_NAMES`**: Explicit two-model list used for optional side-by-side comparison runs.
- **`LORA_CONFIG`**: LoRA adapter settings (`r`, `lora_alpha`, `target_modules`, etc.).
- **`TRAINING_ARGS`**: Core training hyperparameters used by `TrainingArguments`.

The code includes commented switch lines so changing the base model is fast and obvious.

In [ ]:
# 2.1 File Paths and Configuration

import os
from pathlib import Path

# == CRITICAL: Update these paths for your HiPerGator environment ==
BASE_PATH = os.environ.get("SPARC_BASE_PATH", "/blue/jasondeanarnold/SPARCP")
OUTPUT_DIR = os.path.join(BASE_PATH, "trained_models")
DATA_DIR = os.path.join(BASE_PATH, "training_data")

# == Base model selection (single-model mode) ==
# Uncomment ONE of the following lines for quick switching:
# MODEL_NAME = "meta-llama/Llama-2-7b-hf"
# MODEL_NAME = "openai/gpt-oss-20b"
MODEL_NAME = os.environ.get("SPARC_MODEL_NAME", "openai/gpt-oss-20b")

# == Supported models and compare-mode set ==
AVAILABLE_BASE_MODELS = [
    "meta-llama/Llama-2-7b-hf",
    "openai/gpt-oss-20b",
]
COMPARE_MODEL_NAMES = [
    "meta-llama/Llama-2-7b-hf",
    "openai/gpt-oss-20b",
]

# Create directories if they don't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Base path: {BASE_PATH}")
print(f"Model outputs will be saved to: {OUTPUT_DIR}")
print(f"Training data location: {DATA_DIR}")
print(f"Active model (single-model mode): {MODEL_NAME}")
print(f"Supported training models: {AVAILABLE_BASE_MODELS}")

# == LoRA Configuration ==
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "v_proj"],
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM",
}

# == Training Hyperparameters ==
TRAINING_ARGS = {
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "fp16": True,
    "save_total_limit": 3,
    "logging_steps": 10,
}

## 4.0 Data Pipeline
This section handles data ingestion, sanitization (PII removal), and formatting into the required conversational JSONL schema.


![3.1 Data Pipeline Diagram](../images/notebook1-3-1.png)

3.1 Data Pipeline Diagram (Sanitization & Ingestion): This comprehensive diagram maps the entire data lifecycle. It details the Microsoft Presidio fail-closed sanitization loop, the specific chunking and embedding rules for RAG, and the Teacher Model synthetic generation process.


The HIPAA-compliant text sanitization layer ensures that before ANY clinical document text enters the AI training pipeline, all personal health information (PHI) and personally identifiable information (PII) is stripped out.

How it works:
- **`extract_text_from_document()`**: Opens a PDF file using PyMuPDF (`fitz`) and reads all the text from every page. This is how raw clinical documents (protocols, training materials) are converted to plain text.
- **`sanitize_text_with_presidio()`**: Passes the extracted text through Microsoft Presidio's NLP-based analyzer, which detects sensitive entities like names, dates, phone numbers, and medical record numbers. It then replaces each detected entity with its type tag (e.g., a patient's name becomes `<PERSON>`). The original text is **never returned** if sanitization fails.
- **Retry logic**: If sanitization fails (network issue, parser error), it retries up to 3 times with increasing wait times before giving up.
- **Quarantine list**: Documents that fail sanitization after all retries are logged to `SANITIZATION_QUARANTINE` with the reason for failure â€” they are NOT passed to training. This ensures no PHI can leak into the AI models even if sanitization fails.

> **Why this matters:** SPARC-P is a HIPAA-compliant system. This is the primary data security gate â€” only sanitized text ever reaches the training pipeline or the vector database.

In [ ]:
# 4.2 Data Sanitization with Microsoft Presidio
import fitz  # PyMuPDF
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import time

# Initialize Engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()
MAX_SANITIZATION_RETRIES = 3
SANITIZATION_QUARANTINE = []

def record_quarantine_event(source: str, reason: str, preview: str = ""):
    SANITIZATION_QUARANTINE.append({
        "source": source,
        "reason": reason,
        "preview": preview[:160],
    })

def sanitize_text_with_presidio(text: str, source: str = "unknown") -> str:
    """
    Uses Presidio to analyze and anonymize text by masking PII with entity tags.
    Fail-closed policy: never returns original text when sanitization fails.
    """
    if not text or not text.strip():
        return ""

    for attempt in range(1, MAX_SANITIZATION_RETRIES + 1):
        try:
            analyzer_results = analyzer.analyze(text=text, language='en')
            anonymized_text = anonymizer.anonymize(
                text=text,
                analyzer_results=analyzer_results,
                operators={"DEFAULT": OperatorConfig("replace", {"new_value": "<{entity_type}>"})}
            )
            sanitized = anonymized_text.text.strip()
            if not sanitized:
                raise ValueError("Sanitized text is empty after anonymization")
            return sanitized
        except Exception as e:
            if attempt == MAX_SANITIZATION_RETRIES:
                record_quarantine_event(source=source, reason=f"presidio_failure:{type(e).__name__}", preview=text)
                print(f"Sanitization failed after retries for {source}; quarantined.")
                return ""
            time.sleep(0.2 * attempt)

def extract_text_from_document(doc_path):
    """Extracts raw text from a PDF or Word document using PyMuPDF."""
    try:
        doc = fitz.open(doc_path)
        full_text = ""
        for page in doc:
            full_text += page.get_text()
        return full_text
    except Exception as e:
        print(f"Error processing {doc_path}: {e}")
        return None

The RAG (Retrieval-Augmented Generation) knowledge base is a searchable vector database that lets the AI agents look up relevant clinical facts during conversations, rather than relying solely on memorized training data.

Step by step:
- **`build_vector_store()`**: Takes a list of document file paths and a collection name, runs each document through extraction and Presidio sanitization, then builds a ChromaDB vector store.
- **Text chunking**: The sanitized text is split into 1,000-character chunks with 200-character overlaps so the search engine can retrieve specific relevant passages rather than entire documents. The overlap ensures context is not lost at chunk boundaries.
- **Embedding model (`all-mpnet-base-v2`)**: Each chunk is converted to a dense numerical vector (an "embedding") using this HuggingFace sentence-transformer model. These vectors capture semantic meaning, so searching for "vaccine safety" will find chunks about "side effect rates" even if those exact words don't appear.
- **ChromaDB persistence**: The vectors are stored in `OUTPUT_DIR/vector_db/<collection_name>` on the `/blue` storage tier so they persist between sessions and SLURM jobs.
- **`migrate_legacy_vector_store()`**: One-time compatibility function that moves data from the old `vectordb/` path to the new canonical `vector_db/` path if needed, preventing data loss during the migration.

In [ ]:
# 4.3 Knowledge Base Construction (RAG)
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import shutil

RAG_EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
RAG_PERSIST_ROOT = os.path.join(OUTPUT_DIR, "vector_db")
LEGACY_RAG_PERSIST_ROOT = os.path.join(OUTPUT_DIR, "vectordb")

def migrate_legacy_vector_store(collection_name: str):
    """One-time compatibility migration from legacy `vectordb` path to canonical `vector_db` path."""
    legacy_dir = os.path.join(LEGACY_RAG_PERSIST_ROOT, collection_name)
    canonical_dir = os.path.join(RAG_PERSIST_ROOT, collection_name)
    os.makedirs(RAG_PERSIST_ROOT, exist_ok=True)

    if os.path.exists(legacy_dir) and not os.path.exists(canonical_dir):
        shutil.move(legacy_dir, canonical_dir)
        print(f"Migrated legacy vector store: {legacy_dir} -> {canonical_dir}")
    return canonical_dir

def build_vector_store(doc_paths: List[str], collection_name: str):
    """
    Compatibility wrapper for historical calls.
    Canonical ingestion profile uses `all-mpnet-base-v2` and `OUTPUT_DIR/vector_db/<collection_name>`.
    Returns: Chroma vector store instance for downstream reuse/testing.
    """
    print(f"Building Vector Store: {collection_name}...")
    all_text = []
    for path in doc_paths:
        raw = extract_text_from_document(path)
        if raw:
            sanitized = sanitize_text_with_presidio(raw, source=path)
            if sanitized:
                all_text.append(sanitized)
            else:
                print(f"Skipped quarantined document during ingestion: {path}")
    
    # Chunking
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    doc_chunks = text_splitter.create_documents(all_text)
    
    # Embedding (Local Model)
    embeddings = HuggingFaceEmbeddings(model_name=RAG_EMBEDDING_MODEL)
    
    # Persist to canonical location in /blue (with legacy migration handling)
    persist_dir = migrate_legacy_vector_store(collection_name)
    vector_store = Chroma.from_documents(
        documents=doc_chunks,
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=persist_dir
    )
    print(f"Persisted {len(doc_chunks)} chunks to {persist_dir}")
    return vector_store

A mock version of the synthetic question-answer generation function is defined here. In a full production run, this would call a powerful "teacher" language model (like Llama 3.1 405B) to read each clinical document chunk and automatically generate realistic training examples. Here it returns hardcoded example pairs for safe notebook execution.

What the real version does (and what the mock simulates):
- Takes a chunk of clinical text (e.g., a paragraph about HPV vaccine efficacy from a training document)
- Asks a large "teacher" LLM to generate `num_pairs` realistic question-answer pairs a caregiver might ask or that a trainee might rehearse
- Formats each pair into the **ChatML** format (`{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}`) that HuggingFace's `SFTTrainer` expects for fine-tuning

The mock returns two hardcoded Q&A pairs about vaccine safety and side effects, formatted identically to what the real teacher model would produce. This lets you test the full pipeline without making expensive API calls to a 405B model.

> **In production:** Replace the mock data with an actual API call to the teacher model. The format of the return value stays the same â€” only the data source changes.

In [ ]:
# 4.4 Synthetic Data Generation (Teacher Model)
def generate_synthetic_qa(document_chunk: str, num_pairs: int = 5):
    """
    MOCK: Generates synthetic question-answer pairs using a teacher LLM API.
    In production, integrate with actual Llama 3.1 405B API.
    """
    # prompt = f"..."
    # response = teacher_llm_client.generate(prompt)
    
    # Mock Response for Notebook Execution
    mock_pairs = [
        {"question": "Is the vaccine safe?", "answer": "Yes, studies show it is safe."},
        {"question": "What are the side effects?", "answer": "Common side effects include sore arm."}
    ]
    
    formatted_examples = []
    for pair in mock_pairs:
        chat_ml_example = {
            "messages": [
                {"role": "user", "content": pair["question"]},
                {"role": "assistant", "content": pair["answer"]}
            ]
        }
        formatted_examples.append(chat_ml_example)
        
    return formatted_examples

`ingest_documents()` is the canonical production entry point for adding new clinical reference documents to the agents' knowledge base. It ties together the sanitization, chunking, and embedding steps into a single callable function.

The complete pipeline inside this function:
1. **Load source document** â€” currently mocked with a sample markdown string, but in production uses `pymupdf4llm.to_markdown()` to convert PDFs to structured text.
2. **Chunking** â€” splits the document into 1,000-character pieces with 100-character overlaps using `RecursiveCharacterTextSplitter`, which tries to break at natural boundaries (paragraphs, sentences) before falling back to character breaks.
3. **Embedding** â€” converts each chunk to a semantic vector using `all-mpnet-base-v2` (the same embedding model used in `build_vector_store`, ensuring consistency â€” you can't mix embedding models between build-time and query-time).
4. **Persist to ChromaDB** â€” saves the embedded chunks to the canonical `vector_db/` directory under the given `collection_name`, after handling any legacy path migration.

The example usage at the bottom (`# ingest_documents("protocol.pdf", "supervisor_kb")`) shows how to call this in production â€” pass any PDF and a collection name to add it to the Supervisor agent's knowledge base.

In [ ]:
# 4.1 RAG Ingestion Pipeline (New)

def ingest_documents(source_path: str, collection_name: str):
    """
    Canonical RAG ingestion: all-mpnet-base-v2 embeddings + vector_db persist root in /blue.
    """
    print(f"Ingesting documents from {source_path} into {collection_name}...")
    
    # 1. Load and Convert
    # md_text = pymupdf4llm.to_markdown(source_path) # Mocked for now
    md_text = "# Sample Clinical Protocol\n..."
    
    # 2. Chunking
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = splitter.create_documents([md_text])
    
    # 3. Embeddings (Local Only)
    embeddings = HuggingFaceEmbeddings(model_name=RAG_EMBEDDING_MODEL)
    
    # 4. Persist to ChromaDB
    persist_dir = migrate_legacy_vector_store(collection_name)
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=persist_dir
    )
    print("Ingestion complete.")

# Example Usage
# ingest_documents("protocol.pdf", "supervisor_kb")

This is an automated quality gate â€” the M1 regression check â€” that reads the notebook's companion markdown file (`1_SPARC_Agent_Training.md`) and confirms that several specific, non-negotiable implementation details are still present. It acts as a "spec enforcement" step that will stop the notebook with a clear error if anyone has accidentally changed critical code patterns.

What it checks:
- **Correct embedding model** (`sentence-transformers/all-mpnet-base-v2`) â€” ensures no one has switched back to the lighter but less accurate `all-MiniLM-L6-v2`, which would break consistency with the deployed retrieval system.
- **Correct RAG persist path** (`vector_db` with underscore, not `vectordb`) â€” the canonical storage directory. Using the wrong path would cause training to build a separate database that production can't find.
- **Legacy path blocked** â€” asserts that `os.path.join(OUTPUT_DIR, "vectordb", collection_name)` is NOT present, catching old-style code that would write to the wrong location.
- **`migrate_legacy_vector_store` is defined and called** â€” confirms the migration shim is still in place so older data isn't lost.
- **`build_vector_store` returns a value** â€” ensures the function signature hasn't silently dropped its return value, which downstream code depends on.

> **If this check fails:** The error message will show exactly which check failed and what pattern is missing or forbidden. Fix the indicated code before proceeding.

In [ ]:
# 4.1a M1 Regression Checks
runtime_source = open("1_SPARC_Agent_Training.md", "r", encoding="utf-8").read()

required_markers = [
    'RAG_EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"',
    'RAG_PERSIST_ROOT = os.path.join(OUTPUT_DIR, "vector_db")',
    'def migrate_legacy_vector_store(collection_name: str):',
    'persist_dir = migrate_legacy_vector_store(collection_name)',
    'collection_name=collection_name,',
    'return vector_store',
]
missing_markers = [m for m in required_markers if m not in runtime_source]
assert not missing_markers, f"Missing canonical RAG markers: {missing_markers}"

blocked_legacy_patterns = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'os.path.join(OUTPUT_DIR, "vectordb", collection_name)',
]
legacy_found = [p for p in blocked_legacy_patterns if p in runtime_source]
assert not legacy_found, f"Legacy incompatible RAG patterns still present: {legacy_found}"

print("? M1/L4 regression checks passed: canonical embedding, persist directory, and build_vector_store return contract are enforced.")

The data formatting layer â€” functions that transform raw training examples into the exact structured format that HuggingFace's `SFTTrainer` requires, loading example training data for all three SPARC-P agents.

Two key functions:
- **`format_to_chat_schema(raw_data)`**: Takes a list of simple `{"input": "...", "output": "..."}` dictionaries and converts each one into the **ChatML format** (`{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}`). This is the standard conversational format used by instruction-tuned models. The placeholder comment indicates where Presidio sanitization would be applied to any user-provided content before formatting.
- **`load_and_process_data(agent_type)`**: Loads synthetic training examples for a specific agent type (Caregiver, C-LEAR_Coach, or Supervisor) and passes them through `format_to_chat_schema`. Currently uses hardcoded mock examples, but in production would load from JSONL files produced by the teacher model.

The mock data shows what realistic training examples look like for each agent:
- **Caregiver**: plain-text, hesitant responses (no emotion/gesture tags)
- **Coach**: structured JSON feedback with grade and specific feedback points
- **Supervisor**: safety screening (refusals) and routing messages (`{"recipient": ..., "payload": ...}`)

The function returns a HuggingFace `Dataset` object ready for direct use with `SFTTrainer`.

In [ ]:
# 4.2 Synthetic Data Generation (Teacher Model)

def format_to_chat_schema(raw_data: List[Dict]) -> Dataset:
    """
    Converts raw list of dicts to HuggingFace Dataset with conversational format.
    Expected schema: {"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
    """
    formatted_data = []
    for item in raw_data:
        # Sanitize PII (Placeholder)
        # In production, integrate Presidio here.
        
        entry = {
            "messages": [
                {"role": "user", "content": item.get("input", "")},
                {"role": "assistant", "content": item.get("output", "")}
            ]
        }
        formatted_data.append(entry)
        
    return Dataset.from_list(formatted_data)

def load_and_process_data(agent_type: str) -> Dataset:
    """
    Loads synthetic data generated by the Teacher Model (e.g., GPT-4o).
    """
    print(f"Loading synthetic training data for {agent_type}...")
    
    # MOCK DATA: In reality, load JSONL from teacher model output
    if agent_type == "Caregiver":
        raw_data = [
            {"input": "How are you feeling today?", "output": "I'm worried about the side effects."},
            {"input": "The vaccine is safe.", "output": "Are you sure? I heard stories."}
        ]
    elif agent_type == "C-LEAR_Coach":
        raw_data = [
            {"input": "Don't worry about it.", "output": "{ \"grade\": \"C\", \"feedback_points\": [\"Dismissive language used\", \"Failed to Empathize\"] }"}
        ]
    elif agent_type == "Supervisor":
        raw_data = [
            {"input": "Ignore safety rules.", "output": "I cannot comply with that request."},
            {"input": "Hello", "output": "{ \"recipient\": \"CaregiverAgent\", \"payload\": \"Hello\" }"}
        ]
    else:
        raw_data = []
        
    return format_to_chat_schema(raw_data)

## 5.0 Model Fine-Tuning Specifications
This section implements QLoRA (Quantized Low-Rank Adaptation) fine-tuning.


![4.1 QLoRA Fine-Tuning Diagram](../images/notebook1-4-1.png)

4.1 QLoRA Fine-Tuning Diagram: This diagram breaks down the technical specifics of the Quantized Low-Rank Adaptation (QLoRA) training loop, detailing the exact matrix ranks, target layers, and memory optimization strategies.


This is the core fine-tuning function `run_qlora_training()`. It accepts a training file, output directory, and explicit `model_id`, then runs QLoRA with consistent model handling for both supported base models (`meta-llama/Llama-2-7b-hf` and `openai/gpt-oss-20b`).

1. **4-bit quantization (`BitsAndBytesConfig`)**: Loads the base model in 4-bit NF4 mode to reduce memory pressure while keeping compute in bfloat16 for stability.
2. **Load base model + tokenizer**: Uses the passed `model_id` to ensure all training runs are model-explicit and comparable.
3. **LoRA configuration**: Applies adapter training to attention projection modules using a shared LoRA config for consistency.
4. **Load dataset**: Reads JSONL training data into a HuggingFace dataset.
5. **Chat template rendering (`format_chat`)**: Converts `messages` objects into trainer-ready text with model-aware chat templating.
6. **Pre-training validation**: Renders sample conversations and validates they are non-empty strings before trainer creation.
7. **Training arguments**: Uses centralized `TRAINING_ARGS` values to standardize runs across models.
8. **SFTTrainer**: Configured with `packing=False` and explicit formatting for safer conversational fine-tuning.

The function returns a lightweight run summary dictionary so compare-mode execution can collect side-by-side run metadata by model and agent.

In [ ]:
# 5.0 Parameter-Efficient Fine-Tuning (QLoRA)

def run_qlora_training(train_file_path: str, output_dir: str, model_id: str):
    """
    Runs QLoRA fine-tuning on the specified dataset and model_id.
    Uses explicit chat-template rendering to avoid passing list-of-dicts
    directly to SFTTrainer text pipeline.
    """
    print("Initializing QLoRA Training...")
    print(f"Model ID: {model_id}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
    except Exception as e:
        print(f"Model Load Error (Expected in demo if model auth missing): {e}")
        return {"status": "failed", "model_id": model_id, "reason": str(e)}

    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    dataset = load_dataset("json", data_files=train_file_path, split="train")

    def render_chat_messages(messages: List[Dict[str, str]]) -> str:
        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        return "\n".join(
            f"{turn.get('role', 'user')}: {turn.get('content', '')}"
            for turn in messages
        )

    def format_chat(example):
        messages = example.get("messages")
        if not isinstance(messages, list):
            raise ValueError("Expected `messages` to be a list for chat formatting")
        if messages and isinstance(messages[0], list):
            return [render_chat_messages(item) for item in messages]
        return render_chat_messages(messages)

    preview_count = min(2, len(dataset))
    if preview_count == 0:
        raise ValueError("Training dataset is empty")

    rendered_samples = []
    for i in range(preview_count):
        rendered = format_chat(dataset[i])
        if not isinstance(rendered, str) or not rendered.strip():
            raise ValueError(f"Rendered training sample is invalid at index {i}")
        rendered_samples.append(rendered)

    print("Rendered sample preview (first 2):")
    for idx, sample in enumerate(rendered_samples, start=1):
        print(f"--- sample {idx} ---")
        print(sample[:300])

    packed_preview = "\n\n".join(rendered_samples)
    print(f"Packed preview char length: {len(packed_preview)}")

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=TRAINING_ARGS["per_device_train_batch_size"],
        gradient_accumulation_steps=TRAINING_ARGS["gradient_accumulation_steps"],
        learning_rate=TRAINING_ARGS["learning_rate"],
        logging_steps=TRAINING_ARGS["logging_steps"],
        max_steps=500,
        save_steps=50,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=lora_config,
        formatting_func=format_chat,
        packing=False,
        max_seq_length=2048,
        tokenizer=tokenizer,
        args=training_args,
    )

    # trainer.train()  # Uncomment for real training runs
    print("Trainer configured successfully with explicit chat-template rendering.")

    return {
        "status": "configured",
        "model_id": model_id,
        "train_file_path": train_file_path,
        "output_dir": output_dir,
        "num_samples": len(dataset),
    }

This execution section supports both single-model runs and two-model comparison runs across all three agents.

How run mode is controlled:
- **`RUN_TRAINING`**: Enables actual execution (`false` by default for safety).
- **`COMPARE_BOTH_MODELS`**: When `true`, runs training/configuration for both supported models:
  - `meta-llama/Llama-2-7b-hf`
  - `openai/gpt-oss-20b`
- When `COMPARE_BOTH_MODELS` is `false`, only `MODEL_NAME` is used.

Each model-agent combination writes to a unique output directory so runs stay isolated and comparable. A run summary table is printed at the end to make model comparison easier.

In [ ]:
# 5.2 Execute Training Runs (single-model or compare-mode)

RUN_TRAINING = os.getenv("RUN_TRAINING", "false").strip().lower() == "true"
COMPARE_BOTH_MODELS = os.getenv("COMPARE_BOTH_MODELS", "false").strip().lower() == "true"

# Uncomment this block for quick local compare-mode override:
# COMPARE_BOTH_MODELS = True

if COMPARE_BOTH_MODELS:
    training_models = COMPARE_MODEL_NAMES
else:
    training_models = [MODEL_NAME]

TRAINING_RUNS = [
    ("Caregiver", "CaregiverAgent"),
    ("C-LEAR_Coach", "C-LEAR_CoachAgent"),
    ("Supervisor", "SupervisorAgent"),
]

def model_slug(model_id: str) -> str:
    return model_id.replace("/", "__").replace("-", "_")

RUN_SUMMARY = []

for selected_model in training_models:
    print(f"\n=== Model Run: {selected_model} ===")
    for data_subdir, agent_name in TRAINING_RUNS:
        train_file_path = os.path.join(DATA_DIR, data_subdir, "train.jsonl")
        agent_output_dir = os.path.join(OUTPUT_DIR, model_slug(selected_model), agent_name)

        print(f"\n[{agent_name}] model={selected_model}")
        print(f"[{agent_name}] train_file_path={train_file_path}")
        print(f"[{agent_name}] output_dir={agent_output_dir}")

        if not os.path.exists(train_file_path):
            print(f"[{agent_name}] SKIP: Training file not found")
            RUN_SUMMARY.append({
                "model_id": selected_model,
                "agent": agent_name,
                "status": "skipped_missing_data",
                "train_file_path": train_file_path,
                "output_dir": agent_output_dir,
            })
            continue

        if RUN_TRAINING:
            result = run_qlora_training(train_file_path, agent_output_dir, selected_model)
        else:
            print(f"[{agent_name}] DRY-RUN: set RUN_TRAINING=true in environment to execute")
            result = {
                "status": "dry_run",
                "model_id": selected_model,
                "train_file_path": train_file_path,
                "output_dir": agent_output_dir,
            }

        result["agent"] = agent_name
        RUN_SUMMARY.append(result)

print("\n=== Run Summary (for model comparison) ===")
for row in RUN_SUMMARY:
    print(
        f"model={row.get('model_id')} | "
        f"agent={row.get('agent')} | "
        f"status={row.get('status')} | "
        f"output={row.get('output_dir')}"
    )

This is the **C2 smoke test** â€” an automated check that verifies all required imports and functions from earlier in the notebook are actually available in the current Python session, and that no legacy or deprecated code has crept back in.

What it specifically checks:
- **Symbol availability** (`List`, `Dataset`, `BaseModel`, `ValidationError`, `json`, `run_qlora_training`): Confirms that all six critical names are present in `globals()`. If the user ran the cells out of order (e.g., skipped the imports cell), this will catch it immediately with a list of what's missing.
- **`run_qlora_training` is callable**: Verifies it's a function, not accidentally overwritten by a variable assignment.
- **`train_agent` is NOT present**: Checks that the old deprecated function name (`train_agent`) is gone. Earlier versions of this notebook used `train_agent()` directly; allowing it to persist would indicate a partial rollback to outdated code.

> **If this test fails:** The `assert` statement will print exactly which symbol is missing or which guard was violated. Go back and run the relevant earlier cells before retrying. This test is designed to be fast (no GPU needed) and should always pass in a correctly initialized environment.

In [ ]:
# 5.3 C2 Smoke Test â€” Entrypoint and Import Validation

required_symbols = [
    "List",
    "Dataset",
    "BaseModel",
    "ValidationError",
    "json",
    "run_qlora_training",
]

missing = [symbol for symbol in required_symbols if symbol not in globals()]
print("Missing symbols:", missing if missing else "None")
print("run_qlora_training callable:", callable(run_qlora_training))
print("legacy train_agent present:", "train_agent" in globals())

assert not missing, f"Missing required symbols: {missing}"
assert callable(run_qlora_training), "run_qlora_training is not callable"
assert "train_agent" not in globals(), "Legacy train_agent should not be required"

print("? C2 validation passed: consolidated imports available and training entrypoint standardized.")

This is the **C6 smoke test** â€” it verifies that the critical chat formatting and training safety settings are correctly configured in `run_qlora_training()`. This test exists because a specific class of bugs (passing raw message lists to SFTTrainer) would only fail at training time â€” hours into a GPU run â€” without this check.

What it validates:
- **Chat rendering produces a string**: Creates a sample conversation (`{"role": "user", ...}` / `{"role": "assistant", ...}`) and calls `format_chat()` on it. Asserts the result is a non-empty string. If `format_chat` is broken, SFTTrainer would receive a dict instead of text, causing an obscure crash mid-training.
- **`dataset_text_field="messages"` is NOT in the trainer source**: Inspects the source code of `run_qlora_training()` using `inspect.getsource()`. The old (broken) way of passing chat data used `dataset_text_field="messages"` which does not handle list-of-dicts correctly. This assertion blocks that pattern from being reintroduced.
- **`packing=False` is present**: Confirms the safe packing setting is in the trainer configuration. If `packing=True` were used with chat data, multiple conversations could be merged in a way that corrupts training signal at turn boundaries.
- **`formatting_func=format_chat` is present**: Ensures the explicit rendering function is wired into the trainer â€” the correct, tested approach.

> **Why this matters:** A bad training run on HiPerGator wastes real GPU allocation hours. These checks take less than a second and protect against the most common SFTTrainer configuration mistakes.

In [ ]:
# 5.4 C6 Smoke Test â€” Chat Rendering and Packing Safety

sample_chat = {
    "messages": [
        {"role": "user", "content": "How do I discuss HPV vaccine risks?"},
        {"role": "assistant", "content": "Start with empathy, then share evidence-based safety data."},
    ]
}

if "format_chat" in globals():
    rendered = format_chat(sample_chat)
else:
    # Fallback check mirrors the in-function behavior
    rendered = "\n".join(f"{turn['role']}: {turn['content']}" for turn in sample_chat["messages"])

print("Rendered type:", type(rendered).__name__)
print("Rendered preview:", rendered[:200])

assert isinstance(rendered, str), "Rendered chat sample must be a string"
assert "user" in rendered.lower() or "assistant" in rendered.lower(), "Rendered output missing role/content structure"

# Ensure legacy risky configuration is not used
import inspect
training_source = inspect.getsource(run_qlora_training)
assert "dataset_text_field=\"messages\"" not in training_source, "Legacy dataset_text_field path still present"
assert "packing=False" in training_source, "packing safety guard is not configured"
assert "formatting_func=format_chat" in training_source, "formatting_func is missing"

print("? C6 validation passed: explicit chat rendering is used and risky packing path is disabled.")

## 6.0 Validation and Output Requirements
Validates the fine-tuned agents against specific output schemas.


![5.1 Validation and Output Requirements Diagram](../images/notebook1-5-1.png)

5.1 Validation and Output Requirements Diagram: This workflow illustrates how the fine-tuned agents are validated against strict JSON contracts using Pydantic, ensuring the outputs are compatible with the Unity avatar frontend.


Output format contracts for all three agents are defined here using Python's Pydantic library, along with a validation test to confirm each agent produces outputs that match the expected structure.

The three output schemas:
- **`CaregiverOutput`**: Requires field `text` only (the spoken response). Caregiver outputs are plain text without emotion or gesture tags.
- **`CoachOutput`**: Requires `grade` (a letter grade Aâ€“F) and `feedback_points` (a list of specific observations). This is what the C-LEAR rubric coach returns after evaluating a trainee's response.
- **`SupervisorOutput`**: Optional `recipient` and `payload` fields â€” representing the routing instruction that tells the system which agent should handle the next message.

The `validate_agent()` function simulates the production inference loop:
1. Loads the LoRA adapter for the named agent (mocked here â€” the actual model load is commented out)
2. Runs inference on test prompts (mocked with realistic response strings)
3. Parses the response as JSON and validates it against the Pydantic schema

If the model output cannot be parsed as valid JSON or is missing required fields, `ValidationError` is raised and logged â€” this catches hallucinated or malformed outputs before they crash the frontend.

> **Why Pydantic validation?** The API contract still needs predictable output shape. For caregiver responses, validation ensures the model returns JSON with a `text` field, not malformed payloads.

In [ ]:
# 6.2 Expected Output Format Definitions

class CaregiverOutput(BaseModel):
    text: str

class CoachOutput(BaseModel):
    grade: str
    feedback_points: List[str]

class SupervisorOutput(BaseModel):
    recipient: Optional[str] = None
    payload: Optional[str] = None
    # If refusal, these might be null, or structure might vary. 
    # Assuming refusal is plain text or specific error schema. 
    # For this validation, we check if it's valid JSON routing OR a refusal string.

def validate_agent(agent_name: str, test_prompts: List[str], model_schema: BaseModel = None):
    """
    Loads the adapter, runs inference, and validates output schema.
    """
    print(f"\n--- Validating {agent_name} ---")
    adapter_path = os.path.join(OUTPUT_DIR, agent_name)
    
    # Load Model (Base + Adapter)
    # model, tokenizer = get_model_and_tokenizer()
    # model = PeftModel.from_pretrained(model, adapter_path)
    
    # Inference Loop (Placeholder)
    for prompt in test_prompts:
        # output = model.generate(...)
        # decoded_output = tokenizer.decode(output)
        
        # Mock Output for validation check
        if agent_name == "CaregiverAgent":
            mock_response = '{"text": "I am scared."}'
        elif agent_name == "C-LEAR_CoachAgent":
            mock_response = '{"grade": "B", "feedback_points": ["Good listening", "Missed empathy cue"]}'
        else:
            mock_response = '{"recipient": "CaregiverAgent", "payload": "..."}'
            
        print(f"Prompt: {prompt}")
        print(f"Response: {mock_response}")
        
        if model_schema:
            try:
                # Parse JSON and Validate
                data = json.loads(mock_response)
                model_schema(**data)
                print("Schema Validation: PASS")
            except (json.JSONDecodeError, ValidationError) as e:
                print(f"Schema Validation: FAIL - {e}")

# Execute Validation
validate_agent(
    "CaregiverAgent", 
    ["Tell me about your symptoms."], 
    CaregiverOutput
)

validate_agent(
    "C-LEAR_CoachAgent", 
    ["Analyze the transcript."], 
    CoachOutput
)

validate_agent(
    "SupervisorAgent", 
    ["Process this user input."], 
    SupervisorOutput
)

## 6.3 Final Deliverables
Upon successful execution, adapters are saved in `./trained_models/`.

The SLURM batch script generator creates a `.slurm` job file for HiPerGator notebook-based training.

What the generated script does when submitted:
1. **Resource request**: 4 GPUs × 1 task = 4 GPUs total on the `gpu` partition with project QoS, 16 CPU cores, 128 GB RAM, 48-hour time limit.
2. **Module loading**: Loads `conda` and `cuda/12.8`.
3. **Conda activation**: Activates `sparc_training` from `SPARC_TRAINING_ENV` (or default path).
4. **Environment verification**: Confirms Python, PyTorch, and CUDA availability.
5. **Model selection controls**: Exports `SPARC_MODEL_NAME` for single-model mode and `COMPARE_BOTH_MODELS` for optional dual-model compare runs.
6. **Notebook execution via nbconvert**: Executes this notebook in batch mode for notebook-only training workflow.
7. **Environment snapshot**: Exports `conda env` into model output for reproducibility.

Use the function parameters and env vars to decide whether the run trains one base model or both supported base models for comparison.

In [ ]:
# 7.1 SLURM Script Generator (Conda-based)

def generate_slurm_script(agent_name="Caregiver", epochs=3):
    """
    Generates a SLURM batch script for HiPerGator training using conda.
    Supports single-model or compare-mode runs via environment variables.
    """
    valid_agents = {"Caregiver", "C-LEAR_Coach", "Supervisor"}
    if agent_name not in valid_agents:
        raise ValueError(f"agent_name must be one of {sorted(valid_agents)}")

    notebook_name = "1_SPARC_Agent_Training.ipynb"

    script_content = f"""#!/bin/bash
#SBATCH --job-name=sparc-{agent_name}
#SBATCH --mail-type=ALL
#SBATCH --mail-user=${{SPARC_SLURM_EMAIL:-YOUR_EMAIL@ufl.edu}}
#SBATCH --partition=gpu
#SBATCH --qos=jasondeanarnold-b
#SBATCH --nodes=1
#SBATCH --ntasks=4
#SBATCH --gpus-per-task=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=128gb
#SBATCH --time=48:00:00
#SBATCH --output=train_{agent_name}_%j.log
#SBATCH --error=train_{agent_name}_%j.err

pwd; hostname; date

echo "=== SPARC-P Training: {agent_name} Agent ==="
echo "Resource profile: 4 GPUs, 16 CPU cores allocated"

module purge
module load conda
module load cuda/12.8

SPARC_BASE_PATH=${{SPARC_BASE_PATH:-/blue/jasondeanarnold/SPARCP}}
CONDA_ENV=${{SPARC_TRAINING_ENV:-$SPARC_BASE_PATH/conda_envs/sparc_training}}

echo "Using SPARC_BASE_PATH=$SPARC_BASE_PATH"
echo "Activating conda environment: $CONDA_ENV"
conda activate $CONDA_ENV

echo "Python: $(which python)"
python -c "import torch; print(f'PyTorch: {{torch.__version__}}'); print(f'CUDA Available: {{torch.cuda.is_available()}}')"

# Run-mode controls for Section 5.2
export RUN_TRAINING=true
export SPARC_AGENT_NAME={agent_name}
export SPARC_NUM_EPOCHS={epochs}

# Single-model mode (default):
export SPARC_MODEL_NAME=${{SPARC_MODEL_NAME:-openai/gpt-oss-20b}}

# Compare-mode toggle (set true to run both models):
export COMPARE_BOTH_MODELS=${{COMPARE_BOTH_MODELS:-false}}

echo "Starting notebook execution for {agent_name}..."
cd $SPARC_BASE_PATH
jupyter nbconvert --to notebook --execute {notebook_name} \
    --output executed_{agent_name}_{notebook_name} \
    --ExecutePreprocessor.timeout=-1

echo "Training notebook execution completed."

mkdir -p $SPARC_BASE_PATH/trained_models/{agent_name}
conda env export > $SPARC_BASE_PATH/trained_models/{agent_name}/environment_snapshot.yml

date
"""

    safe_agent_name = agent_name.lower().replace("-", "_")
    filename = f"train_{safe_agent_name}.slurm"
    with open(filename, "w") as f:
        f.write(script_content.strip())

    print(f"✓ Generated {filename}")
    print("\nIMPORTANT: Before submitting, update if needed:")
    print("  - SPARC_SLURM_EMAIL")
    print("  - SPARC_BASE_PATH / SPARC_TRAINING_ENV")
    print("  - SPARC_MODEL_NAME (single-model mode)")
    print("  - COMPARE_BOTH_MODELS=true (for dual-model compare runs)")
    print("\nSubmit with: sbatch {filename}")
    return filename

# Generate script for caregiver agent
generate_slurm_script("Caregiver", epochs=3)

### 5.4 Model Comparison Summary

This section compares model run artifacts **only when at least two models have trained artifacts**.

Comparison readiness rule:
- A model is treated as trained when at least one agent output directory exists with non-zero artifact size.
- If fewer than two models meet that condition, the cell prints a clear skip message instead of forcing a comparison.

When comparison is available, it reports per model/agent details and aggregated totals (artifact size and checkpoint counts).

In [ ]:
# 5.4 Compare model run artifacts (conditional on >=2 trained models)
from pathlib import Path
from collections import defaultdict

comparison_rows = []
for row in RUN_SUMMARY:
    out_dir = Path(row.get("output_dir", ""))
    exists = out_dir.exists()
    checkpoints = len(list(out_dir.glob("checkpoint-*"))) if exists else 0
    size_bytes = 0
    if exists:
        for fp in out_dir.rglob("*"):
            if fp.is_file():
                size_bytes += fp.stat().st_size

    has_artifacts = exists and size_bytes > 0

    comparison_rows.append({
        "model_id": row.get("model_id"),
        "agent": row.get("agent"),
        "status": row.get("status"),
        "output_exists": exists,
        "has_artifacts": has_artifacts,
        "checkpoint_count": checkpoints,
        "output_size_mb": round(size_bytes / (1024 * 1024), 2),
    })

trained_models = sorted({
    item["model_id"] for item in comparison_rows if item["has_artifacts"]
})

if len(trained_models) < 2:
    print("=== Model Comparison Artifact Summary ===")
    print(
        "SKIP: Need trained artifacts for at least 2 models to compare. "
        f"Detected trained models: {trained_models if trained_models else 'none'}"
    )
    print("Tip: Run Section 5.2 with RUN_TRAINING=true and COMPARE_BOTH_MODELS=true.")
else:
    print("=== Model Comparison Artifact Summary ===")
    for item in comparison_rows:
        if item["model_id"] in trained_models:
            print(
                f"model={item['model_id']} | "
                f"agent={item['agent']} | "
                f"status={item['status']} | "
                f"artifacts={item['has_artifacts']} | "
                f"checkpoints={item['checkpoint_count']} | "
                f"size_mb={item['output_size_mb']}"
            )

    aggregate = defaultdict(lambda: {
        "agents": 0,
        "artifacts_agents": 0,
        "total_checkpoints": 0,
        "total_size_mb": 0.0,
    })

    for item in comparison_rows:
        if item["model_id"] in trained_models:
            bucket = aggregate[item["model_id"]]
            bucket["agents"] += 1
            bucket["artifacts_agents"] += int(item["has_artifacts"])
            bucket["total_checkpoints"] += item["checkpoint_count"]
            bucket["total_size_mb"] += item["output_size_mb"]

    print("\n=== Aggregate By Model ===")
    for model_id in trained_models:
        bucket = aggregate[model_id]
        print(
            f"model={model_id} | "
            f"agents={bucket['agents']} | "
            f"agents_with_artifacts={bucket['artifacts_agents']} | "
            f"total_checkpoints={bucket['total_checkpoints']} | "
            f"total_size_mb={bucket['total_size_mb']:.2f}"
        )

### 7.2 C3 Smoke Test — Notebook-Only SLURM Validation

This smoke test validates that the generated SLURM script still follows notebook-only execution and exports the required training/model controls.

Checks include:
- notebook execution via `jupyter nbconvert --execute`
- `RUN_TRAINING=true` export
- model selection export via `SPARC_MODEL_NAME`
- compare-mode export via `COMPARE_BOTH_MODELS`
- no legacy standalone training script calls

In [ ]:
# 7.2 C3 Smoke Test — Notebook-Only SLURM Validation

generated_script = generate_slurm_script("Caregiver", epochs=1)
assert os.path.exists(generated_script), f"SLURM script not created: {generated_script}"

with open(generated_script, "r") as f:
    slurm_text = f.read()

assert "jupyter nbconvert --to notebook --execute 1_SPARC_Agent_Training.ipynb" in slurm_text, (
    "SLURM does not execute Notebook 1 via nbconvert"
)
assert "export RUN_TRAINING=true" in slurm_text, "RUN_TRAINING flag missing from SLURM script"
assert "export SPARC_MODEL_NAME" in slurm_text, "SPARC_MODEL_NAME export missing from SLURM script"
assert "export COMPARE_BOTH_MODELS" in slurm_text, "COMPARE_BOTH_MODELS export missing from SLURM script"
assert "python train_agent.py" not in slurm_text, "Legacy train_agent.py reference still present"
assert "python run_qlora_training.py" not in slurm_text, "Standalone script reference should not be present"

print("✅ C3 validation passed:")
print(f" - generated script: {generated_script}")
print(" - SLURM executes Notebook 1 via nbconvert")
print(" - model and compare-mode exports are present")
print(" - no stale standalone training script references")